In [25]:
import pandas as pd
import numpy as np

In [26]:
#Load dataset
#读取数据

df_raw = pd.read_csv("Region summary_ New South Wales STE 1.csv")

#Create a copy of the original dataset
#创建原始数据副本

df = df_raw.copy()

print(df.head())
print(df.shape)
print(df.columns)
print(df.dtypes)

  Measure Code                                 Parent Description  \
0     ERP_P_20  Estimated resident population - year ended 30 ...   
1       ERP_21  Estimated resident population - year ended 30 ...   
2     ERP_M_20  Estimated resident population - year ended 30 ...   
3     ERP_F_20  Estimated resident population - year ended 30 ...   
4       ERP_19  Estimated resident population - year ended 30 ...   

                                     Description  2011  2015  2016  2017  \
0            Estimated resident population (no.)   NaN   NaN   NaN   NaN   
1               Population density (persons/km2)   NaN   NaN   NaN   NaN   
2    Estimated resident population - males (no.)   NaN   NaN   NaN   NaN   
3  Estimated resident population - females (no.)   NaN   NaN   NaN   NaN   
4                     Median age - males (years)   NaN   NaN   NaN   NaN   

   2018       2019       2020       2021       2022       2023       2024  \
0   NaN  8046748.0  8110610.0  8097062.0  8166704.0

In [27]:
#Standardise column names and check missing values
#标准化列名+检查缺失值

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]", "", regex=True)
)

df = df.replace(r"^\s*$", np.nan, regex=True)

print(df.columns)
print(df.isna().sum())

Index(['measure_code', 'parent_description', 'description', '2011', '2015',
       '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024',
       '2025'],
      dtype='str')
measure_code            0
parent_description      0
description             0
2011                  541
2015                  780
2016                  343
2017                  752
2018                  675
2019                  530
2020                  509
2021                  108
2022                  507
2023                  593
2024                  591
2025                  797
dtype: int64


In [28]:
#Remove duplicated rows and convert year columns
#删除重复行+转换年份

print(df.duplicated().sum())

df = df.drop_duplicates()

year_columns = [col for col in df.columns if col.isdigit()]

for col in year_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=year_columns, how="all")

text_columns = [
    "measure_code",
    "parent_description",
    "description"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print(df.shape)
print(df.dtypes)
print(df.head())

0
(800, 15)
measure_code              str
parent_description        str
description               str
2011                  float64
2015                  float64
2016                  float64
2017                  float64
2018                  float64
2019                  float64
2020                  float64
2021                  float64
2022                  float64
2023                  float64
2024                  float64
2025                  float64
dtype: object
  measure_code                                 parent_description  \
0     ERP_P_20  Estimated resident population - year ended 30 ...   
1       ERP_21  Estimated resident population - year ended 30 ...   
2     ERP_M_20  Estimated resident population - year ended 30 ...   
3     ERP_F_20  Estimated resident population - year ended 30 ...   
4       ERP_19  Estimated resident population - year ended 30 ...   

                                     description  2011  2015  2016  2017  \
0            Estimated resident p

In [29]:
#Convert the dataset from wide format to long format
#将数据从 wide format 转换为 long format

df_long = df.melt(
    id_vars=[
        "measure_code",
        "parent_description",
        "description"
    ],
    value_vars=year_columns,
    var_name="year",
    value_name="value"
)

df_long["year"] = pd.to_numeric(
    df_long["year"],
    errors="coerce"
)

df_long = df_long.dropna(subset=["value"])

print(df_long.head())
print(df_long.shape)
print(df_long.dtypes)

    measure_code                                 parent_description  \
127    CENSUS_34  Aboriginal and Torres Strait Islander Peoples ...   
128     CENSUS_2  Aboriginal and Torres Strait Islander Peoples ...   
140    CENSUS_15                     Religious affiliation - Census   
141    CENSUS_16                     Religious affiliation - Census   
142    CENSUS_17                     Religious affiliation - Census   

                                           description  year     value  
127  Aboriginal and Torres Strait Islander Peoples ...  2011  172620.0  
128  Aboriginal and Torres Strait Islander Peoples (%)  2011       2.5  
140                                       Buddhism (%)  2011       2.9  
141                                   Christianity (%)  2011      64.5  
142                                       Hinduism (%)  2011       1.7  
(2874, 5)
measure_code              str
parent_description        str
description               str
year                    int64
value

In [30]:
# Save cleaned datasets
#保存清理后的数据

df.to_csv(
    "Region_summary_NSW_cleaned_wide.csv",
    index=False
)

df_long.to_csv(
    "Region_summary_NSW_cleaned_long.csv",
    index=False
)


print(df[year_columns].describe())
print(df_long.describe())

               2011          2015          2016          2017          2018  \
count  2.590000e+02  2.000000e+01  4.570000e+02  4.800000e+01  1.250000e+02   
mean   6.735284e+05  8.022549e+06  1.530573e+06  4.820513e+06  2.249978e+06   
std    4.126540e+06  1.919068e+07  7.971771e+06  1.495873e+07  9.359767e+06   
min    1.000000e-01  2.500000e+01  3.000000e-01  5.200000e+00  1.100000e+00   
25%    7.950000e+00  5.945000e+01  8.700000e+00  1.094500e+03  1.125000e+03   
50%    5.990000e+01  4.079050e+04  6.090000e+01  1.431480e+04  4.201200e+04   
75%    6.621400e+04  3.001438e+06  1.271790e+05  6.921535e+05  3.736930e+05   
max    5.356605e+07  8.012986e+07  8.012986e+07  8.012986e+07  8.012986e+07   

               2019          2020          2021          2022          2023  \
count  2.700000e+02  2.910000e+02  6.920000e+02  2.930000e+02  2.070000e+02   
mean   1.133374e+06  1.103433e+06  6.610403e+05  3.916004e+05  2.247894e+05   
std    6.384606e+06  6.072784e+06  4.611798e+06  1.